In [5]:
%pip -q uninstall -y torchao
%pip -q install -r /content/requirements.txt


import re
import numpy as np
import pandas as pd
import torch

from collections import Counter
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit
from catboost import CatBoostClassifier, CatBoostRanker, Pool
from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader

In [6]:
root = Path("/content")

train = pd.read_csv(root / "train.csv")
test = pd.read_csv(root / "test.csv")
sample = pd.read_csv(root / "sample_submission.csv")

print("Train:", train.shape)
print("Test:", test.shape)
print("Запросов в train:", train.query_id.nunique())
print("Запросов в test:", test.query_id.nunique())
print("Общие query_id:", len(set(train.query_id) & set(test.query_id)))
print("Метки:", train.relevance.value_counts().to_dict())
print("Пропуски train:")
print(train.isna().sum())
print("Пропуски test:")
print(test.isna().sum())
print("Размер групп train:")
print(train.groupby("query_id").size().describe())
print("Доля релевантных по источникам:")
print(train.groupby("ingest_source").relevance.agg(["mean", "count"]))

assert train.id.is_unique
assert test.id.is_unique
assert set(train.query_id).isdisjoint(set(test.query_id))
assert set(sample.id) == set(test.id)
assert list(sample.columns) == ["id", "prediction"]

Train: (31530, 12)
Test: (13507, 11)
Запросов в train: 3042
Запросов в test: 1305
Общие query_id: 0
Метки: {0: 23362, 1: 8168}
Пропуски train:
id                          0
query_id                    0
document_id                 0
article_id                  0
query                       0
title                    1405
text                        0
passage_index            1398
article_passage_count    1959
article_char_count       2534
ingest_source               0
relevance                   0
dtype: int64
Пропуски test:
id                          0
query_id                    0
document_id                 0
article_id                  0
query                       0
title                     686
text                        0
passage_index             738
article_passage_count    1035
article_char_count       1275
ingest_source               0
dtype: int64
Размер групп train:
count    3042.000000
mean       10.364892
std         1.072353
min         2.000000
25%        10.000000
5

In [7]:
def ndcg_at_10(frame, predictions):
    result = frame[["id", "query_id", "relevance"]].copy()
    result["prediction"] = np.asarray(predictions)

    result = result.sort_values(
        ["query_id", "prediction", "id"],
        ascending=[True, False, True],
        kind="stable"
    )

    result["position"] = result.groupby("query_id").cumcount() + 1
    result["gain"] = np.where(
        result["position"] <= 10,
        result["relevance"] / np.log2(result["position"] + 1),
        0.0
    )

    dcg = result.groupby("query_id")["gain"].sum()
    positives = result.groupby("query_id")["relevance"].sum().clip(upper=10).astype(int)

    ideal = np.r_[0.0, np.cumsum(1 / np.log2(np.arange(2, 12)))]
    denominators = ideal[positives.to_numpy()]
    scores = np.divide(
        dcg.to_numpy(),
        denominators,
        out=np.zeros(len(dcg)),
        where=denominators > 0
    )

    return float(scores.mean())


splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
fit_idx, val_idx = next(
    splitter.split(train, groups=train["query_id"])
)

print("Запросов для обучения:", train.iloc[fit_idx].query_id.nunique())
print("Запросов для проверки:", train.iloc[val_idx].query_id.nunique())
print(
    "Пересечение:",
    len(
        set(train.iloc[fit_idx].query_id)
        & set(train.iloc[val_idx].query_id)
    )
)

Запросов для обучения: 2433
Запросов для проверки: 609
Пересечение: 0


In [8]:
all_rows = pd.concat(
    [train.drop(columns="relevance"), test],
    ignore_index=True
)

pattern = re.compile(r"[а-яёa-z0-9]+")

def tokenize(value):
    return pattern.findall(str(value).lower())


query_tokens = all_rows["query"].fillna("").map(tokenize).tolist()
title_tokens = all_rows["title"].fillna("").map(tokenize).tolist()
text_tokens = all_rows["text"].fillna("").map(tokenize).tolist()

stopwords = {
    "как", "что", "кто", "где", "когда", "какой", "какая",
    "какие", "сколько", "почему", "чем", "из", "на", "по",
    "в", "и", "о", "от", "до", "а", "с", "за", "у", "это",
    "был", "была", "были", "для", "его", "ее", "их",
    "ли", "же", "или", "не"
}

content_tokens = [
    [word for word in words if word not in stopwords]
    for words in query_tokens
]

features = pd.DataFrame(index=all_rows.index)

targets = {
    "title": title_tokens,
    "text": text_tokens,
    "both": [
        title + text
        for title, text in zip(title_tokens, text_tokens)
    ],
    "lead": [words[:100] for words in text_tokens]
}

for target_name, target_tokens in targets.items():
    for query_name, selected_queries in [
        ("all", query_tokens),
        ("content", content_tokens)
    ]:
        rows = []

        for query_words, document_words in zip(
            selected_queries, target_tokens
        ):
            query_set = set(query_words)
            document_count = Counter(document_words)
            common = query_set.intersection(document_count)

            rows.append([
                len(common),
                len(common) / (len(query_set) + 1e-6),
                sum(document_count[word] for word in query_set),
                sum(document_count[word] for word in query_set)
                / (len(document_words) + 1),
                len(common) / (len(set(document_words)) + 1e-6),
                int(bool(query_set) and query_set.issubset(document_count)),
                min(
                    (document_words.index(word) for word in common),
                    default=10000
                )
            ])

        values = np.asarray(rows, dtype=np.float32)

        for column, position in {
            "hits": 0,
            "recall": 1,
            "freq": 2,
            "density": 3,
            "precision": 4,
            "full": 5,
            "first": 6
        }.items():
            features[f"{target_name}_{query_name}_{column}"] = values[:, position]

In [9]:
for target_name, target_tokens in [
    ("title", title_tokens),
    ("text", text_tokens)
]:
    documents = [" ".join(words[:300]) for words in target_tokens]
    queries = [" ".join(words) for words in query_tokens]
    size = len(documents)

    for analyzer, ngram_range in [
        ("word", (1, 2)),
        ("char", (3, 4))
    ]:
        vectorizer = TfidfVectorizer(
            analyzer=analyzer,
            ngram_range=ngram_range,
            min_df=2,
            max_features=70000,
            dtype=np.float32,
            sublinear_tf=True
        )

        matrix = vectorizer.fit_transform(documents + queries)
        similarities = matrix[:size].multiply(matrix[size:]).sum(axis=1)

        features[f"{target_name}_{analyzer}_tfidf"] = (
            np.asarray(similarities).ravel()
        )

        del matrix, vectorizer

for name, tokens in [
    ("query", query_tokens),
    ("title", title_tokens),
    ("text", text_tokens)
]:
    features[f"{name}_len_words"] = np.asarray(
        [len(words) for words in tokens],
        dtype=np.float32
    )
    features[f"{name}_len_chars"] = (
        all_rows[name].fillna("").str.len().astype(np.float32)
    )

features["passage_index"] = all_rows["passage_index"]
features["article_passage_count"] = all_rows["article_passage_count"]

clean_char_count = (
    all_rows["article_char_count"]
    .fillna("")
    .astype(str)
    .str.replace(r"[^\d,.-]", "", regex=True)
    .str.replace(",", ".", regex=False)
)

features["article_char_count"] = pd.to_numeric(
    clean_char_count,
    errors="coerce"
)

features["passage_position"] = (
    features["passage_index"]
    / (features["article_passage_count"] + 1)
)

features["source"] = (
    all_rows["ingest_source"]
    .map({"S0": 0, "S1": 1, "S2": 2, "S3": 3})
    .fillna(-1)
)

features["q_what"] = (
    all_rows["query"]
    .str.lower()
    .str[:20]
    .str.contains(
        "что|кто|где|когда|сколько|почему|какой|как ",
        regex=True
    )
    .astype(int)
)

features["same_article_n"] = (
    all_rows
    .groupby(["query_id", "article_id"])["article_id"]
    .transform("size")
    .astype(np.float32)
)

relative_columns = [
    "title_word_tfidf",
    "text_word_tfidf",
    "title_char_tfidf",
    "text_char_tfidf",
    "text_content_recall",
    "title_content_recall"
]

for column in relative_columns:
    grouped = features[column].groupby(all_rows["query_id"])

    features[f"{column}_rank"] = grouped.rank(
        ascending=False,
        method="average"
    ).astype(np.float32)

    features[f"{column}_topgap"] = (
        features[column] - grouped.transform("max")
    ).astype(np.float32)

features = (
    features
    .replace([np.inf, -np.inf], np.nan)
    .astype(np.float32)
    .fillna(-1)
)

x_train = features.iloc[:len(train)].copy()
x_test = features.iloc[len(train):].copy()
y = train["relevance"].to_numpy()

assert len(x_train) == len(train)
assert len(x_test) == len(test)
assert not x_train.isna().any().any()

print("Число признаков:", x_train.shape[1])

Число признаков: 85


In [10]:
fit_order = fit_idx[np.argsort(train.query_id.to_numpy()[fit_idx], kind="stable")]
val_order = val_idx[np.argsort(train.query_id.to_numpy()[val_idx], kind="stable")]
y = train.relevance.to_numpy()

passages = (
    all_rows["title"].fillna("").astype(str)
    + ". "
    + all_rows["text"].fillna("").astype(str).str[:2000]
).tolist()
queries = all_rows["query"].fillna("").astype(str).tolist()


def within_query_rank(scores, query_ids):
    frame = pd.DataFrame({
        "query_id": np.asarray(query_ids),
        "score": np.asarray(scores)
    })
    return frame.groupby("query_id")["score"].rank(method="average", pct=True).to_numpy()

val_groups = train.iloc[val_idx].query_id.to_numpy()


In [11]:
assert torch.cuda.is_available(), "Включите GPU в Colab"

bge_model = CrossEncoder(
    "BAAI/bge-reranker-v2-m3",
    max_length=512,
    device="cuda"
)

bge_scores = np.asarray(
    bge_model.predict(
        list(zip(queries, passages)),
        batch_size=4,
        show_progress_bar=True,
        convert_to_numpy=True
    )
).reshape(-1)

assert len(bge_scores) == len(all_rows)
assert np.isfinite(bge_scores).all()

print("BGE отдельно:", ndcg_at_10(train.iloc[val_idx], bge_scores[val_idx]))

del bge_model
torch.cuda.empty_cache()


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

... (2 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Batches:   0%|          | 0/11260 [00:00<?, ?it/s]

BGE отдельно: 0.9258732697971074


In [12]:
features["bge_score"] = bge_scores.astype(np.float32)

bge_group = features["bge_score"].groupby(
    all_rows["query_id"]
)

features["bge_rank"] = bge_group.rank(
    ascending=False,
    method="average"
).astype(np.float32)

features["bge_topgap"] = (
    features["bge_score"]
    - bge_group.transform("max")
).astype(np.float32)

x_train_bge = features.iloc[:len(train)].copy()
x_test_bge = features.iloc[len(train):].copy()

fit_pool_bge = Pool(
    x_train_bge.iloc[fit_order],
    y[fit_order],
    group_id=train.query_id.to_numpy()[fit_order]
)

val_pool_bge = Pool(
    x_train_bge.iloc[val_order],
    y[val_order],
    group_id=train.query_id.to_numpy()[val_order]
)

ranker_bge = CatBoostRanker(
    iterations=1300,
    depth=6,
    learning_rate=0.045,
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    random_seed=42,
    verbose=False
)

ranker_bge.fit(
    fit_pool_bge,
    eval_set=val_pool_bge,
    early_stopping_rounds=120
)

ranker_bge_val = ranker_bge.predict(
    x_train_bge.iloc[val_idx]
)

print("CatBoost с BGE:", ndcg_at_10(train.iloc[val_idx], ranker_bge_val))


CatBoost с BGE: 0.935893059556264


In [13]:
final_val = ranker_bge_val.copy()

full_order = np.argsort(train.query_id.to_numpy(), kind="stable")
full_pool_bge = Pool(
    x_train_bge.iloc[full_order],
    y[full_order],
    group_id=train.query_id.to_numpy()[full_order]
)

full_ranker_bge = CatBoostRanker(
    iterations=max(100, ranker_bge.best_iteration_ + 1),
    depth=6,
    learning_rate=0.045,
    loss_function="YetiRank",
    random_seed=42,
    verbose=False
)

full_ranker_bge.fit(full_pool_bge)
final_prediction = full_ranker_bge.predict(x_test_bge)

print("Базовая проверка:", ndcg_at_10(train.iloc[val_idx], final_val))
print("Прогнозов для теста:", len(final_prediction))


Базовая проверка: 0.935893059556264
Прогнозов для теста: 13507


In [14]:

import gc
import math
import torch.nn.functional as F

from pathlib import Path
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup
)

assert torch.cuda.is_available()

cache_dir = Path("/content/ranking_lora")
cache_dir.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    cache_dir / "bge_zero_shot_scores.npz",
    ids=all_rows.id.to_numpy(),
    scores=bge_scores
)

model_name_lora = "BAAI/bge-reranker-v2-m3"
tokenizer_lora = AutoTokenizer.from_pretrained(model_name_lora)

encoded_train = tokenizer_lora(
    queries[:len(train)],
    passages[:len(train)],
    padding="max_length",
    truncation=True,
    max_length=384,
    return_tensors="pt"
)


def make_pairs(indices):
    pairs = []

    for _, group in train.iloc[indices].groupby(
        "query_id",
        sort=False
    ):
        positive = group.loc[
            group.relevance.eq(1)
        ].index.to_numpy()

        negative = group.loc[
            group.relevance.eq(0)
        ].index.to_numpy()

        if len(positive) == 0 or len(negative) == 0:
            continue

        positive = positive[
            np.argsort(bge_scores[positive])
        ]

        negative = negative[
            np.argsort(-bge_scores[negative])
        ]

        for j in range(min(2, len(negative))):
            pairs.append((
                int(positive[j % len(positive)]),
                int(negative[j])
            ))

    return torch.tensor(
        pairs,
        dtype=torch.long
    )


def make_lora_model():
    base = AutoModelForSequenceClassification.from_pretrained(
        model_name_lora,
        low_cpu_mem_usage=True
    )

    config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["query", "value"],
        modules_to_save=["classifier"],
        task_type=TaskType.SEQ_CLS
    )

    model = get_peft_model(base, config)

    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={
            "use_reentrant": False
        }
    )

    return model.to("cuda")


def predict_lora(model, indices):
    model.eval()
    result = []

    for start in range(0, len(indices), 8):
        selected = indices[start:start + 8]

        batch = tokenizer_lora(
            [queries[i] for i in selected],
            [passages[i] for i in selected],
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to("cuda")

        with torch.inference_mode():
            with torch.autocast(
                "cuda",
                dtype=torch.float16
            ):
                scores = (
                    model(**batch)
                    .logits
                    .float()
                    .view(-1)
                )

        result.append(
            scores.cpu().numpy()
        )

    return np.concatenate(result)


def train_lora(model, indices, epochs, seed):
    pairs = make_pairs(indices)

    assert len(pairs) > 0

    generator = torch.Generator().manual_seed(seed)

    loader = DataLoader(
        pairs,
        batch_size=2,
        shuffle=True,
        generator=generator
    )

    optimizer = torch.optim.AdamW(
        (
            parameter
            for parameter in model.parameters()
            if parameter.requires_grad
        ),
        lr=2e-5,
        weight_decay=0.01
    )

    total_updates = math.ceil(
        len(loader) / 4
    ) * epochs

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(
            1,
            int(total_updates * 0.1)
        ),
        num_training_steps=total_updates
    )

    scaler = torch.amp.GradScaler("cuda")

    model.train()

    for epoch in range(epochs):
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0

        for step, pair in enumerate(loader):
            count = len(pair)

            selected = torch.cat([
                pair[:, 0],
                pair[:, 1]
            ])

            batch = {
                name: values[selected].to("cuda")
                for name, values in encoded_train.items()
            }

            with torch.autocast(
                "cuda",
                dtype=torch.float16
            ):
                logits = (
                    model(**batch)
                    .logits
                    .float()
                    .view(-1)
                )

                loss = F.softplus(
                    -(
                        logits[:count]
                        - logits[count:]
                    )
                ).mean()

            scaler.scale(loss / 4).backward()

            running_loss += float(
                loss.detach().cpu()
            )

            if (
                (step + 1) % 4 == 0
                or step + 1 == len(loader)
            ):
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    (
                        parameter
                        for parameter in model.parameters()
                        if parameter.requires_grad
                    ),
                    1.0
                )

                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        print(
            "Эпоха:",
            epoch + 1,
            "попарная потеря:",
            running_loss / len(loader)
        )

        yield epoch + 1
        model.train()

In [15]:
base_val_lora = within_query_rank(
    final_val,
    val_groups
)

base_score_lora = ndcg_at_10(
    train.iloc[val_idx],
    base_val_lora
)

selection_lora = (
    base_score_lora,
    0,
    0.0
)
best_lora_val = base_val_lora.copy()

lora_model = make_lora_model()

for epoch in train_lora(
    lora_model,
    fit_idx,
    epochs=2,
    seed=42
):
    lora_val = within_query_rank(
        predict_lora(lora_model, val_idx),
        val_groups
    )

    for weight in np.arange(0.1, 1.01, 0.1):
        mixed = (
            (1 - weight) * base_val_lora
            + weight * lora_val
        )

        score = ndcg_at_10(
            train.iloc[val_idx],
            mixed
        )

        if score > selection_lora[0]:
            selection_lora = (
                score,
                epoch,
                float(weight)
            )
            best_lora_val = lora_val.copy()

            lora_model.save_pretrained(
                cache_dir / "best_validation_adapter"
            )

    print(
        "После эпохи",
        epoch,
        "лучший NDCG@10:",
        selection_lora[0]
    )

best_score_lora, best_epoch_lora, weight_lora = (
    selection_lora
)

print("Текущее решение:", base_score_lora)
print("Лучшее решение с LoRA:", best_score_lora)
print("Эпоха:", best_epoch_lora)
print("Вес LoRA:", weight_lora)

del lora_model
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Эпоха: 1 попарная потеря: 0.5886861063193206
После эпохи 1 лучший NDCG@10: 0.935893059556264
Эпоха: 2 попарная потеря: 0.4895441995355191
После эпохи 2 лучший NDCG@10: 0.9365730128059715
Текущее решение: 0.935893059556264
Лучшее решение с LoRA: 0.9365730128059715
Эпоха: 2
Вес LoRA: 0.30000000000000004


In [16]:
previous_prediction = np.asarray(final_prediction).copy()

if weight_lora > 0 and best_score_lora > base_score_lora:
    final_lora_model = make_lora_model()

    for _ in train_lora(
        final_lora_model,
        np.arange(len(train)),
        epochs=best_epoch_lora,
        seed=42
    ):
        pass

    final_lora_model.save_pretrained(
        cache_dir / "full_training_adapter"
    )

    test_indices_lora = np.arange(
        len(train),
        len(all_rows)
    )

    lora_test_scores = predict_lora(
        final_lora_model,
        test_indices_lora
    )

    np.save(
        cache_dir / "lora_test_scores.npy",
        lora_test_scores
    )

    final_prediction = (
        (1 - weight_lora)
        * within_query_rank(
            previous_prediction,
            test.query_id
        )
        + weight_lora
        * within_query_rank(
            lora_test_scores,
            test.query_id
        )
    )

    del final_lora_model
    gc.collect()
    torch.cuda.empty_cache()

    print("LoRA добавлена с весом:", weight_lora)
    print(
        "Изменено прогнозов:",
        np.count_nonzero(
            np.abs(final_prediction - previous_prediction) > 1e-12
        ),
        "из",
        len(test)
    )
else:
    final_prediction = previous_prediction
    print("LoRA не улучшила валидацию")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Эпоха: 1 попарная потеря: 0.5635767485938283
Эпоха: 2 попарная потеря: 0.47819072386388256
LoRA добавлена с весом: 0.30000000000000004
Изменено прогнозов: 13507 из 13507


In [17]:
current_val = (
    (1 - weight_lora) * base_val_lora
    + weight_lora * best_lora_val
)
current_score = ndcg_at_10(train.iloc[val_idx], current_val)
bge_val_rank = within_query_rank(bge_scores[val_idx], val_groups)

results = []
for weight in [0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    mixed = (1 - weight) * current_val + weight * bge_val_rank
    results.append((ndcg_at_10(train.iloc[val_idx], mixed), weight))

best_score_extra, best_weight_extra = max(results)
print("Текущая проверка:", current_score)
print("Лучшая проверка с дополнительным BGE:", best_score_extra)
print("Вес дополнительного BGE:", best_weight_extra)

if best_score_extra > current_score + 0.001:
    final_prediction = (
        (1 - best_weight_extra)
        * within_query_rank(final_prediction, test.query_id)
        + best_weight_extra
        * within_query_rank(bge_scores[len(train):], test.query_id)
    )
    print("Дополнительный BGE добавлен")
else:
    print("Дополнительный BGE не улучшил проверку")


Текущая проверка: 0.9365730128059715
Лучшая проверка с дополнительным BGE: 0.9365730128059715
Вес дополнительного BGE: 0
Дополнительный BGE не улучшил проверку


In [18]:
import gc
from pathlib import Path
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

assert torch.cuda.is_available(), "Включите GPU в Colab"

gc.collect()
torch.cuda.empty_cache()

giga_model = SentenceTransformer(
    "ai-sage/Giga-Embeddings-instruct-480M-0826",
    trust_remote_code=True,
    device="cuda",
    model_kwargs={"torch_dtype": torch.float16}
)

giga_model.max_seq_length = 512

print("GigaEmbeddings загружена")


modules.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/25.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/55.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.56k [00:00<?, ?B/s]

configuration_gigarembed.py:   0%|          | 0.00/150 [00:00<?, ?B/s]

... (9 outputs left out)

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.00 [00:00<?, ?B/s]

GigaEmbeddings загружена


In [ ]:
def predict_giga(row_positions, filename):
    row_positions = np.asarray(row_positions, dtype=int)
    cache_path = Path("/content") / filename
    scores = np.full(len(row_positions), np.nan, dtype=np.float32)

    if cache_path.exists():
        saved = np.load(cache_path)
        if saved.shape == scores.shape:
            scores = saved.astype(np.float32)

    if np.isfinite(scores).all():
        return scores

    selected_queries = np.asarray(queries, dtype=str)[row_positions]
    unique_queries, inverse = np.unique(selected_queries, return_inverse=True)
    instruction = "Given a web search query, retrieve relevant passages that answer the query"
    prompted_queries = [
        f"Instruct: {instruction}\nQuery: {query}"
        for query in unique_queries
    ]

    query_embeddings = giga_model.encode(
        prompted_queries,
        batch_size=8,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True
    ).astype(np.float32)

    for start in tqdm(range(0, len(row_positions), 512)):
        end = min(start + 512, len(row_positions))
        if np.isfinite(scores[start:end]).all():
            continue

        documents = [passages[index] for index in row_positions[start:end]]
        document_embeddings = giga_model.encode(
            documents,
            batch_size=8,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        ).astype(np.float32)

        scores[start:end] = np.sum(
            query_embeddings[inverse[start:end]] * document_embeddings,
            axis=1
        )
        np.save(cache_path, scores)

    assert np.isfinite(scores).all()
    return scores


base_val_giga = (
    (1 - weight_lora) * base_val_lora
    + weight_lora * best_lora_val
)

if best_score_extra > current_score + 0.001:
    base_val_giga = (
        (1 - best_weight_extra) * base_val_giga
        + best_weight_extra * within_query_rank(bge_scores[val_idx], val_groups)
    )

base_score_giga = ndcg_at_10(train.iloc[val_idx], base_val_giga)
giga_val_scores = predict_giga(val_idx, "giga_480m_validation.npy")
giga_val_rank = within_query_rank(giga_val_scores, val_groups)

giga_results = []
for weight in [0.0, 0.05, 0.1, 0.2, 0.3, 0.4]:
    mixed = (1 - weight) * base_val_giga + weight * giga_val_rank
    giga_results.append((ndcg_at_10(train.iloc[val_idx], mixed), weight))

best_giga_score, best_giga_weight = max(giga_results)

print("Текущая валидация:", base_score_giga)
print("GigaEmbeddings отдельно:", ndcg_at_10(train.iloc[val_idx], giga_val_scores))
print("Лучшая валидация с GigaEmbeddings:", best_giga_score)
print("Вес GigaEmbeddings:", best_giga_weight)
print("Прирост:", best_giga_score - base_score_giga)


In [20]:
giga_applied = best_giga_score > base_score_giga + 0.002

if giga_applied:
    test_positions_giga = np.arange(len(train), len(all_rows))
    giga_test_scores = predict_giga(test_positions_giga, "giga_480m_test.npy")
    final_prediction = (
        (1 - best_giga_weight) * within_query_rank(final_prediction, test.query_id)
        + best_giga_weight * within_query_rank(giga_test_scores, test.query_id)
    )
    print("GigaEmbeddings добавлена к прогнозу")
else:
    print("GigaEmbeddings не улучшила валидацию достаточно; прогноз сохранён")


Batches:   0%|          | 0/164 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]

GigaEmbeddings добавлена к прогнозу


In [ ]:
assert giga_applied
assert len(giga_val_scores) == len(val_idx)
assert len(giga_test_scores) == len(test)

fit_giga_scores = predict_giga(
    fit_idx,
    "giga_480m_fit.npy"
)

giga_scores_all = np.empty(
    len(all_rows),
    dtype=np.float32
)

giga_scores_all[fit_idx] = fit_giga_scores
giga_scores_all[val_idx] = giga_val_scores
giga_scores_all[len(train):] = giga_test_scores

assert np.isfinite(giga_scores_all).all()

features_giga = features.copy()
features_giga["giga_score"] = giga_scores_all

giga_group = features_giga["giga_score"].groupby(
    all_rows["query_id"]
)

features_giga["giga_rank"] = giga_group.rank(
    ascending=False,
    method="average"
).astype(np.float32)

features_giga["giga_topgap"] = (
    features_giga["giga_score"]
    - giga_group.transform("max")
).astype(np.float32)

features_giga["giga_center"] = (
    features_giga["giga_score"]
    - giga_group.transform("mean")
).astype(np.float32)

features_giga["rank_disagreement"] = (
    features_giga["giga_rank"]
    - features_giga["bge_rank"]
).astype(np.float32)

x_train_giga = features_giga.iloc[:len(train)].copy()
x_test_giga = features_giga.iloc[len(train):].copy()

fit_pool_giga = Pool(
    x_train_giga.iloc[fit_order],
    y[fit_order],
    group_id=train.query_id.to_numpy()[fit_order]
)

val_pool_giga = Pool(
    x_train_giga.iloc[val_order],
    y[val_order],
    group_id=train.query_id.to_numpy()[val_order]
)

ranker_giga = CatBoostRanker(
    iterations=1000,
    depth=5,
    learning_rate=0.045,
    l2_leaf_reg=5,
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    random_seed=2026,
    verbose=False
)

ranker_giga.fit(
    fit_pool_giga,
    eval_set=val_pool_giga,
    early_stopping_rounds=120
)

ranker_giga_val = ranker_giga.predict(
    x_train_giga.iloc[val_idx]
)

current_val_giga = (
    (1 - best_giga_weight) * base_val_giga
    + best_giga_weight * giga_val_rank
)

current_score_giga = ndcg_at_10(
    train.iloc[val_idx],
    current_val_giga
)

ranker_giga_val_rank = within_query_rank(
    ranker_giga_val,
    val_groups
)

experiments_giga = []

for weight in [0.0, 0.1, 0.25, 0.4, 0.6, 1.0]:
    candidate_val = (
        (1 - weight) * current_val_giga
        + weight * ranker_giga_val_rank
    )

    experiments_giga.append((
        ndcg_at_10(train.iloc[val_idx], candidate_val),
        weight
    ))

best_score_giga_cat, best_weight_giga_cat = max(
    experiments_giga
)

print("Текущая валидация:", current_score_giga)
print(
    "CatBoost с Giga отдельно:",
    ndcg_at_10(train.iloc[val_idx], ranker_giga_val)
)
print("Лучшее сочетание:", best_score_giga_cat)
print("Вес нового CatBoost:", best_weight_giga_cat)
print("Прирост:", best_score_giga_cat - current_score_giga)

In [22]:
use_giga_catboost = (
    best_weight_giga_cat > 0
    and best_score_giga_cat > current_score_giga
)

if use_giga_catboost:
    full_pool_giga = Pool(
        x_train_giga.iloc[full_order],
        y[full_order],
        group_id=train.query_id.to_numpy()[full_order]
    )

    full_ranker_giga = CatBoostRanker(
        iterations=max(100, ranker_giga.best_iteration_ + 1),
        depth=5,
        learning_rate=0.045,
        l2_leaf_reg=5,
        loss_function="YetiRank",
        random_seed=2026,
        verbose=False
    )

    full_ranker_giga.fit(full_pool_giga)

    output_prediction = (
        (1 - best_weight_giga_cat)
        * within_query_rank(final_prediction, test.query_id)
        + best_weight_giga_cat
        * within_query_rank(
            full_ranker_giga.predict(x_test_giga),
            test.query_id
        )
    )
else:
    output_prediction = np.asarray(final_prediction).copy()

submission_giga_catboost = sample[["id"]].merge(
    pd.DataFrame({
        "id": test["id"],
        "prediction": output_prediction
    }),
    on="id",
    how="left",
    validate="one_to_one"
)

assert submission_giga_catboost.shape == (len(test), 2)
assert submission_giga_catboost.columns.tolist() == ["id", "prediction"]
assert submission_giga_catboost.id.is_unique
assert submission_giga_catboost.prediction.notna().all()
assert np.isfinite(
    submission_giga_catboost.prediction.to_numpy()
).all()

submission_giga_catboost.to_csv(
    "/content/submission.csv",
    index=False
)

print("Файл сохранён: /content/submission.csv")
print("Новый CatBoost добавлен:", use_giga_catboost)
print(submission_giga_catboost.head())

Файл сохранён: /content/submission.csv
Новый CatBoost добавлен: True
                       id  prediction
0  r_00100bd94cf8513c6500    0.600000
1  r_0011d1a81cda1b1c720f    0.400000
2  r_00135550b108ebd88432    0.636364
3  r_001cac3cb24c86e57ef0    0.100000
4  r_001e8de6d4886f85b86e    0.300000
